# NLI Lyrics Corpus Builder (v2)

**Goal:** Build a corpus of English-language songs by artists with a known L1, for Native Language Identification research.

**Scope:** Phases 2–6 of the data collection plan.
- **Phase 2** — Build candidate artist pool (Wikidata + MusicBrainz)
- **Phase 3** — Pre-filter candidates (genre, Wikipedia presence, optional year)
- **Phase 4** — Scrape lyrics from Genius (with error handling)
- **Phase 5** — Language filtering with fastText
- **Phase 6** — Metadata filtering (covers, duplicates)

Manual L1 verification (Phase 7) happens after this notebook on the produced `artists_for_verification.csv`.

**New additions (v2)**
- Tightened genre filter: dropped generic `latin` keyword (was matching bolero/folk artists who don't write in English)
- Added optional year filter to drop pre-1970s artists
- Scrape function wrapped in try/except so one bad artist doesn't kill the whole run
- Plain `tqdm` (not `tqdm.notebook`) to avoid `ipywidgets` dependency issues

## Setup

### Install dependencies

Run once:

```bash
pip install lyricsgenius requests musicbrainzngs pandas fasttext-wheel tqdm python-dotenv
```

### API credentials

You need a **Genius API token** (required for Phase 4). Get one here:
- Go to https://genius.com/api-clients
- Sign up / log in, click "New API Client"
- Fill in any app name and website (e.g., `http://example.com`)
- Copy the **Client Access Token**

Create a `.env` file in the same folder as this notebook:

```
GENIUS_TOKEN=your_token_here
```

**Make sure `.env` is in your `.gitignore`** before committing anything.

In [6]:
import os
import json
import time
import re
from pathlib import Path
from collections import Counter

import pandas as pd
import requests
from tqdm import tqdm  # Plain tqdm (no ipywidgets dependency)
from dotenv import load_dotenv

load_dotenv()

GENIUS_TOKEN = os.getenv("GENIUS_TOKEN")
assert GENIUS_TOKEN, "Set GENIUS_TOKEN in .env before running Phase 4"

print("Genius token loaded")

import warnings
warnings.filterwarnings("ignore")

Genius token loaded


## Configuration — change this block for each L1

Everything downstream reads from `CONFIG`. To swap languages, edit this cell and re-run.

In [7]:
CONFIG = {
    # L1 label used in output files
    "L1": "spanish",

    # ISO 3166-1 alpha-2 country codes where this L1 is dominant
    "countries": [
        "ES", "MX", "AR", "CO", "PE", "CL", "VE", "EC", "GT", "CU",
        "DO", "HN", "BO", "SV", "PY", "NI", "CR", "PR", "PA", "UY",
    ],

    # fastText language code for this L1 (for detecting/excluding it in lyrics)
    "l1_fasttext_code": "es",

    # Wikidata occupation QIDs to include
    # rapper, hip-hop musician, singer, singer-songwriter
    "occupation_qids": ["Q2252262", "Q177220", "Q753110", "Q488205"],

    # Genre keywords for Phase 3 filter
    # Note: "latin" deliberately excluded, it matches all Spanish-language genres
    # including bolero, ranchera, etc., which rarely have English releases
    "target_genres": [
        "hip hop", "hip-hop", "rap", "trap", "reggaeton",
        "urban", "r&b", "rnb", "drill", "grime", "dembow",
    ],

    # Optional: exclude artists born before this year (None to disable)
    # 1970 cuts most pre-modern-pop artists who didn't write English-language tracks
    "min_birth_year": 1970,

    # Phase 5 thresholds
    "min_english_ratio": 0.5,
    "borderline_ratio": 0.3,
    "min_lines_per_song": 8,

    # Phase 8 requirement 
    "min_songs_per_artist": 1,

    # Output structure
    "data_dir": Path("./data"),
}

DATA = CONFIG["data_dir"] / CONFIG["L1"]
for sub in ["raw", "interim", "processed", "logs"]:
    (DATA / sub).mkdir(parents=True, exist_ok=True)

print(f"Workspace: {DATA.resolve()}")
print(f"L1: {CONFIG['L1']} | Countries: {len(CONFIG['countries'])} | fastText code: {CONFIG['l1_fasttext_code']}")

Workspace: /Users/nataliajimenez/Desktop/AUC/TM/project/data/spanish
L1: spanish | Countries: 20 | fastText code: es


---
# Phase 2 — Candidate artist pool

## 2A — Wikidata SPARQL

In [13]:
def iso_to_wikidata_country(iso_codes):
    """Map ISO country codes to Wikidata QIDs."""
    mapping = {
        "ES": "Q29",  "MX": "Q96",  "AR": "Q414", "CO": "Q739", "PE": "Q419",
        "CL": "Q298", "VE": "Q717", "EC": "Q736", "GT": "Q774", "CU": "Q241",
        "DO": "Q786", "HN": "Q783", "BO": "Q750", "SV": "Q792", "PY": "Q733",
        "NI": "Q811", "CR": "Q800", "PR": "Q1183","PA": "Q804", "UY": "Q77",
        "IT": "Q38",  "NL": "Q55",  "BE": "Q31",  "RU": "Q159", "BY": "Q184",
        "UA": "Q212", "KZ": "Q232", "US": "Q30",  "GB": "Q145", "CA": "Q16",
        "AU": "Q408", "IE": "Q27",  "NZ": "Q664",
    }
    return [mapping[c] for c in iso_codes if c in mapping]


def build_wikidata_query(country_qids, occupation_qids):
    countries_block = " ".join(f"wd:{q}" for q in country_qids)
    occupations_block = " ".join(f"wd:{q}" for q in occupation_qids)
    return f"""
    SELECT DISTINCT ?artist ?artistLabel ?birthPlaceLabel ?birthYear ?genreLabel ?enwiki ?genius
    WHERE {{
      ?artist wdt:P31 wd:Q5 ;
              wdt:P106 ?occupation ;
              wdt:P27 ?country .
      VALUES ?occupation {{ {occupations_block} }}
      VALUES ?country {{ {countries_block} }}
      OPTIONAL {{ ?artist wdt:P19 ?birthPlace . }}
      OPTIONAL {{ ?artist wdt:P569 ?birthDate . BIND(YEAR(?birthDate) AS ?birthYear) }}
      OPTIONAL {{ ?artist wdt:P136 ?genre . }}
      OPTIONAL {{ ?artist wdt:P2373 ?genius . }}
      OPTIONAL {{
        ?enwiki schema:about ?artist ;
                schema:isPartOf <https://en.wikipedia.org/> .
      }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en" . }}
    }}
    LIMIT 5000
    """


def query_wikidata(sparql_query):
    url = "https://query.wikidata.org/sparql"
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "NLI-research-notebook/1.0 (academic, AUC text mining course)",
    }
    response = requests.get(url, params={"query": sparql_query}, headers=headers, timeout=90)
    response.raise_for_status()
    return response.json()


country_qids = iso_to_wikidata_country(CONFIG["countries"])
query = build_wikidata_query(country_qids, CONFIG["occupation_qids"])
print(f"Querying Wikidata for {len(country_qids)} countries × {len(CONFIG['occupation_qids'])} occupations...")

raw = query_wikidata(query)
bindings = raw["results"]["bindings"]
print(f"Returned {len(bindings)} rows (may include duplicates due to multi-genre artists)")

Querying Wikidata for 20 countries × 4 occupations...
Returned 5000 rows (may include duplicates due to multi-genre artists)


In [14]:
def parse_wikidata_results(bindings):
    rows = []
    for b in bindings:
        rows.append({
            "wikidata_id": b["artist"]["value"].rsplit("/", 1)[-1],
            "name": b.get("artistLabel", {}).get("value", ""),
            "birthplace": b.get("birthPlaceLabel", {}).get("value", ""),
            "birth_year": b.get("birthYear", {}).get("value", ""),
            "genre": b.get("genreLabel", {}).get("value", ""),
            "enwiki_url": b.get("enwiki", {}).get("value", ""),
            "genius_slug": b.get("genius", {}).get("value", ""),
        })
    df = pd.DataFrame(rows)
    # Collapse multi-genre duplicates: keep one row per artist, join genres
    df = (
        df.groupby(["wikidata_id", "name", "birthplace", "birth_year", "enwiki_url", "genius_slug"],
                   dropna=False, as_index=False)
          .agg({"genre": lambda s: "; ".join(sorted({g for g in s if g}))})
    )
    df["source"] = "wikidata"
    return df


candidates_wd = parse_wikidata_results(bindings)
candidates_wd.to_csv(DATA / "interim" / "candidates_wikidata.csv", index=False)
print(f"Wikidata: {len(candidates_wd)} unique artists")
candidates_wd.head(10)

Wikidata: 4128 unique artists


,wikidata_id,name,birthplace,birth_year,enwiki_url,genius_slug,genre,source
0,Q1000682,Fernando Carrillo,Caracas,1965,https://en.wikipedia.org/wiki/Fernando_Carrillo,,,wikidata
1,Q1000682,Fernando Carrillo,Caracas,1970,https://en.wikipedia.org/wiki/Fernando_Carrillo,,,wikidata
2,Q1001328,Buddy Richard,Graneros,1943,https://en.wikipedia.org/wiki/Buddy_Richard,,pop music; rock music,wikidata
3,Q1001328,Buddy Richard,Iquique,1943,https://en.wikipedia.org/wiki/Buddy_Richard,,pop music; rock music,wikidata
4,Q1003303,Fran Perea,Málaga,1978,https://en.wikipedia.org/wiki/Fran_Perea,Fran-perea,,wikidata
5,Q1007183,Gabriel Coronel,Barquisimeto,1987,https://en.wikipedia.org/wiki/Gabriel_Coronel,,,wikidata
6,Q1007206,Gabriel Porras,Mexico City,1968,https://en.wikipedia.org/wiki/Gabriel_Porras,,,wikidata
7,Q10284993,Francisco Martino,,1884,,,,wikidata
8,Q10287032,Félix B. Caignet,San Luis,1892,https://en.wikipedia.org/wiki/F%C3%A9lix_B._Ca...,,,wikidata
9,Q10293131,Gustavo Nápoli,Buenos Aires,1967,,,rock music,wikidata


## 2B — MusicBrainz

In [15]:
import musicbrainzngs

musicbrainzngs.set_useragent("NLI-research-notebook", "1.0", "academic@auc.nl")


def fetch_musicbrainz_artists(country_codes, tags=("hip hop", "rap", "reggaeton"), limit=100):
    rows = []
    for cc in tqdm(country_codes, desc="MusicBrainz countries"):
        for tag in tags:
            query = f'country:{cc} AND tag:"{tag}"'
            try:
                result = musicbrainzngs.search_artists(query=query, limit=limit)
            except musicbrainzngs.WebServiceError as e:
                print(f"  {cc}/{tag}: error {e}")
                time.sleep(2)
                continue
            for a in result.get("artist-list", []):
                rows.append({
                    "musicbrainz_id": a.get("id"),
                    "name": a.get("name"),
                    "country": a.get("country"),
                    "tags": "; ".join(t["name"] for t in a.get("tag-list", [])) if a.get("tag-list") else "",
                    "disambiguation": a.get("disambiguation", ""),
                    "source": "musicbrainz",
                })
            time.sleep(1.0)  # MusicBrainz requires 1 req/sec
    return pd.DataFrame(rows).drop_duplicates(subset="musicbrainz_id")


candidates_mb = fetch_musicbrainz_artists(CONFIG["countries"])
candidates_mb.to_csv(DATA / "interim" / "candidates_musicbrainz.csv", index=False)
print(f"MusicBrainz: {len(candidates_mb)} unique artists")
candidates_mb.head(10)

MusicBrainz countries: 100%|████████████████████████████████████████████████████████████| 20/20 [01:26<00:00,  4.33s/it]

MusicBrainz: 621 unique artists


,musicbrainz_id,name,country,tags,disambiguation,source
0,cb15675b-24a4-4092-8554-61f03b6017d8,Cookin’ Soul,ES,lo-fi hip hop; instrumental hip hop; instrumen...,"Spanish hip hop producer, formerly trio/duo",musicbrainz
1,3aa0cc15-37d0-4c30-bb76-5d6e5b81c474,SFDK,ES,hip hop,,musicbrainz
2,ff734df1-6795-43ec-84a7-ae04b356dad1,Nach,ES,hip hop,Spanish MC,musicbrainz
3,f7625f34-1799-42a4-a1c6-22b5377c767b,Arca,ES,trip-hop; electronic; pop; industrial; ambient...,Venezuelan producer & composer,musicbrainz
4,925d3918-c112-47b5-b799-fd62c4477578,Juaninacka,ES,hip hop,,musicbrainz
5,94c82c5a-60a6-4dd4-8412-0624ab8c0d92,ToteKing,ES,hip hop,,musicbrainz
6,43706fe2-bf24-4b1b-8d0b-28f8439b3d55,Violadores del Verso,ES,hip hop,also known as Doble V,musicbrainz
7,40cab918-cea6-4932-9a55-86fe57f24427,Falsalarma,ES,hip hop,,musicbrainz
8,e4459963-7a10-4b15-812b-356db2ceea32,El Meswy,ES,hip hop,,musicbrainz
9,3ea3967e-0055-4018-8496-da4115035aa7,Hazhe,ES,hip hop,Spanish hip hop producer,musicbrainz


## 2C — Merge candidates from both sources

In [16]:
def normalize_name(s):
    if pd.isna(s):
        return ""
    s = s.lower().strip()
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s


wd = candidates_wd.copy()
mb = candidates_mb.copy()
wd["name_norm"] = wd["name"].apply(normalize_name)
mb["name_norm"] = mb["name"].apply(normalize_name)

merged = wd.merge(mb, on="name_norm", how="outer", suffixes=("_wd", "_mb"))
merged["name"] = merged["name_wd"].fillna(merged["name_mb"])
merged["sources"] = merged.apply(
    lambda r: "; ".join([s for s in [r.get("source_wd"), r.get("source_mb")] if pd.notna(s)]),
    axis=1,
)

candidates = merged[["name", "name_norm", "sources", "wikidata_id", "musicbrainz_id",
                     "birthplace", "birth_year", "genre", "country", "tags",
                     "enwiki_url", "genius_slug", "disambiguation"]].drop_duplicates("name_norm")

candidates.to_csv(DATA / "interim" / "candidates_merged.csv", index=False)
print(f"Merged candidates: {len(candidates)}")
print(f"  Wikidata only:    {(merged['source_wd'].notna() & merged['source_mb'].isna()).sum()}")
print(f"  MusicBrainz only: {(merged['source_mb'].notna() & merged['source_wd'].isna()).sum()}")
print(f"  Both sources:     {(merged['source_wd'].notna() & merged['source_mb'].notna()).sum()}")

Merged candidates: 4220
  Wikidata only:    4094
  MusicBrainz only: 593
  Both sources:     34


---
# Phase 3 — Pre-filter

Aggressive filtering before scraping. Each filter is named so you can see what dropped what.

**Why these filters:**
1. **Genre filter** — only keep hip-hop/rap/urban genres where English-language tracks are common. Generic "latin" tag deliberately excluded.
2. **Wikipedia filter** — require an English Wikipedia page. This is a proxy for "documented artist with verifiable bio", which we'll need anyway for L1 verification (Phase 7).
3. **Year filter (optional)** — drop pre-1970 artists who are unlikely to have English-language modern hip-hop releases.

In [17]:
# load data if not first time running
candidates = pd.read_csv(DATA / "interim" / "candidates_merged.csv")
print(f"Loaded {len(candidates)} existing candidates from disk")

Loaded 4220 existing candidates from disk


In [18]:
pre_filtered = candidates.copy()
print(f"Starting candidates:        {len(pre_filtered)}")

# Filter 1: name not empty
pre_filtered = pre_filtered[pre_filtered["name"].str.len() > 0]
print(f"After name filter:          {len(pre_filtered)}")

# Filter 2: relevant genre (hip-hop/rap/urban, not generic 'latin')
def has_target_genre(row):
    genres = " ".join(str(x).lower() for x in [row.get("genre", ""), row.get("tags", "")])
    return any(kw in genres for kw in CONFIG["target_genres"])

pre_filtered = pre_filtered[pre_filtered.apply(has_target_genre, axis=1)]
print(f"After genre filter:         {len(pre_filtered)}")

# Filter 3: must have an English Wikipedia page
pre_filtered = pre_filtered[
    pre_filtered["enwiki_url"].notna() & (pre_filtered["enwiki_url"] != "")
]
print(f"After Wikipedia filter:     {len(pre_filtered)}")

# Filter 4 (optional): birth year cutoff
if CONFIG["min_birth_year"] is not None:
    # Convert birth_year to numeric (some are strings, some empty)
    pre_filtered["birth_year_num"] = pd.to_numeric(pre_filtered["birth_year"], errors="coerce")
    # Keep rows where birth_year is unknown (NaN) OR >= cutoff
    mask = pre_filtered["birth_year_num"].isna() | (pre_filtered["birth_year_num"] >= CONFIG["min_birth_year"])
    pre_filtered = pre_filtered[mask]
    print(f"After year filter (>={CONFIG['min_birth_year']}):  {len(pre_filtered)}")

pre_filtered.to_csv(DATA / "interim" / "candidates_prefiltered.csv", index=False)

n = len(pre_filtered)
est_min = n * 30 / 60
print(f"\nEstimated scrape time: {est_min:.0f} min ({est_min/60:.1f} hours) at max_songs=25")

if n < 100:
    print("Good, proceed with scraping")
elif n < 300:
    print("Acceptable, proceed but be patient")
else:
    print(f"Still too many ({n}). Consider tightening CONFIG['target_genres'] or raising min_birth_year.")

print("\nSample of survivors (do these names look like artists who would write English songs?):")
pre_filtered[["name", "country", "birth_year", "genre", "tags"]].head(20)

Starting candidates:        4220
After name filter:          4220
After genre filter:         674
After Wikipedia filter:     73
After year filter (>=1970):  64

Estimated scrape time: 32 min (0.5 hours) at max_songs=25
Good, proceed with scraping

Sample of survivors (do these names look like artists who would write English songs?):


,name,country,birth_year,genre,tags
60,Aid Alonso,NaN,1990.0,hip-hop,NaN
108,Aldo Ranks,PA,1973.0,reggae en Español,latin; latin urban; reggaeton
211,Amandititita,NaN,1979.0,cumbia; reggaeton,NaN
236,Ana Brenda Contreras,NaN,1986.0,contemporary R&B,NaN
257,Ana Tijoux,CL,1977.0,hip-hop,hip-hop; hip hop; chile; female vocals; rapper
336,"Antonio ""El Potro"" Álvarez",NaN,1979.0,Latin pop; merengue; reggaeton; tropical music,NaN
340,Antonio José,NaN,1995.0,Latin pop; ballade; flamenco; pop music; pop r...,NaN
374,Arca,ES,1989.0,experimental music; neoperreo,trip-hop; electronic; pop; industrial; ambient...
383,Arianna Puello,ES,1977.0,rapping,hip hop
424,Babo,MX,1976.0,Mexican hip-hop,hip hop


In [19]:
pre_filtered[["name", "country", "birth_year", "genre"]].to_string()

'                            name country  birth_year                                                                                                           genre\n60                    Aid Alonso     NaN      1990.0                                                                                                         hip-hop\n108                   Aldo Ranks      PA      1973.0                                                                                               reggae en Español\n211                 Amandititita     NaN      1979.0                                                                                               cumbia; reggaeton\n236         Ana Brenda Contreras     NaN      1986.0                                                                                                contemporary R&B\n257                   Ana Tijoux      CL      1977.0                                                                                                         hip-hop\n336

By looking at this output I have decided to remove artists which may add noise by having ghostwriters and artists that are not native spanish speakers (remove/change for other languages)

In [20]:
# Drop high ghostwriter risk before scraping
ghostwriter_risk = [
    "J Balvin", "Ricky Martin", "Thalía", "Paulina Rubio",
    "Tini Stoessel", "Camilo", "Danna", "Lele Pons", "Yasmin Deliz",
    "Antonio José", "Beatriz Luengo", "Fonseca", "Erika Ender",
    "Diana Neira", "Ana Brenda Contreras", "María Isabel López",
    "Mayra Goñi", "Wendy Sulca",
    # Foreign-born or US-raised, will fail L1:
    "Farid Bang", "J.R. Writer", "Zack de la Rocha",
]

before = len(pre_filtered)
pre_filtered = pre_filtered[~pre_filtered["name"].isin(ghostwriter_risk)]
print(f"Dropped {before - len(pre_filtered)} ghostwriter-risk / non-L1 artists")
print(f"Remaining: {len(pre_filtered)}")

Dropped 20 ghostwriter-risk / non-L1 artists
Remaining: 44


---
# Phase 4 — Scrape lyrics from Genius

**Checkpointed:** each artist's scrape saves to `data/<L1>/raw/<artist_slug>.json`. If the loop crashes or you interrupt it, re-running skips artists already done.

**Wrapped in try/except:** one bad artist won't kill the whole run.

Firts we perform a pre-scrapping sanity check:

In [7]:
print(f"About to scrape {len(pre_filtered)} artists")
print(f"Estimated time: {len(pre_filtered) * 30 / 60:.0f} minutes")
print("\nFirst 10:")
print(pre_filtered[["name", "country"]].head(10).to_string())
print("\nLast 10:")
print(pre_filtered[["name", "country"]].tail(10).to_string())

About to scrape 41 artists
Estimated time: 20 minutes

First 10:
                           name country
65                   Aid Alonso     NaN
119                  Aldo Ranks      PA
233                Amandititita     NaN
284                  Ana Tijoux      CL
365  Antonio "El Potro" Álvarez     NaN
406                        Arca      ES
417              Arianna Puello      ES
462                        Babo      MX
562                   Bocafloja     NaN
620                    CandyMan      CU

Last 10:
               name country
3453  Osmani García      CU
3528   Papi Sánchez      DO
3546   Pato Machete     NaN
3695          Porta     NaN
3702         P-Star     NaN
3727       Q5932808     NaN
3860         Reykon     NaN
3933          Rocca     NaN
4370      Tote King     NaN
4572  Yotuel Romero     NaN


In [8]:
import lyricsgenius

genius = lyricsgenius.Genius(
    GENIUS_TOKEN,
    remove_section_headers=True,
    skip_non_songs=True,
    excluded_terms=["(Remix)", "(Live)", "(Acoustic)", "(Demo)", "(Instrumental)"],
    timeout=30,
    retries=3,
)
genius.verbose = False  # Set as attribute, not constructor arg (compatibility fix)
print("Genius client initialized")

Genius client initialized


In [19]:
def slugify(name):
    s = re.sub(r"[^\w\s-]", "", str(name).lower()).strip()
    return re.sub(r"[-\s]+", "_", s)


def scrape_artist(name, max_songs=25):
    """Scrape discography; return dict with songs or error.
    Compatible with lyricsgenius 3.12+ which uses to_dict() for IDs.
    """
    try:
        artist = genius.search_artist(name, max_songs=max_songs, sort="popularity")
    except Exception as e:
        return {"error": str(e), "name": name}
    if artist is None:
        return {"error": "not_found", "name": name}
    
    # Get artist ID from to_dict
    try:
        artist_id = artist.to_dict().get("id")
    except Exception:
        artist_id = None
    
    songs = []
    for song in artist.songs:
        try:
            song_dict = song.to_dict()
            song_id = song_dict.get("id")
        except Exception:
            song_id = None
        
        # featured_artists handling - check if it's a list of dicts or objects
        featured = []
        try:
            for fa in (song.featured_artists or []):
                if isinstance(fa, dict):
                    featured.append(fa.get("name", ""))
                elif hasattr(fa, "name"):
                    featured.append(fa.name)
        except Exception:
            pass
        
        songs.append({
            "song_id": song_id,
            "title": song.title,
            "lyrics": song.lyrics or "",
            "url": song.url,
            "featured_artists": featured,
        })
    
    return {"name": name, "genius_id": artist_id, "songs": songs}


def scrape_all(candidates_df, max_songs=25):
    """Loop over candidates with checkpointing and error handling."""
    raw_dir = DATA / "raw"
    log_path = DATA / "logs" / "scrape_log.jsonl"

    for _, row in tqdm(candidates_df.iterrows(), total=len(candidates_df), desc="Scraping Genius"):
        slug = slugify(row["name"])
        if not slug:
            continue
        out_path = raw_dir / f"{slug}.json"
        if out_path.exists():
            continue  # Checkpoint: skip already scraped

        # Wrap entire scrape attempt so one bad artist doesn't kill the run
        try:
            result = scrape_artist(row["name"], max_songs=max_songs)
        except Exception as e:
            result = {"error": f"unexpected: {e}", "name": row["name"]}

        try:
            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False, indent=2)
            with open(log_path, "a", encoding="utf-8") as f:
                f.write(json.dumps({
                    "name": row["name"],
                    "slug": slug,
                    "status": "error" if "error" in result else "ok",
                    "n_songs": len(result.get("songs", [])),
                }) + "\n")
        except Exception as e:
            print(f"Failed to save {slug}: {e}")

        time.sleep(1.0)  # Be polite to Genius

print("Scrape function ready.")

Scrape function ready.


In [29]:
# Kick off the scrape. This can take a while - check the time estimate above.
# If you interrupt, re-running this cell resumes from where it stopped (checkpointed).
scrape_all(pre_filtered, max_songs=100)
print("Scrape complete")

Scraping Genius: 100%|███████████████████████████████████████████████████████████████| 41/41 [1:29:32<00:00, 131.04s/it]

Scrape complete


In [21]:
# Load all scraped data into a song-level DataFrame
def load_scraped_songs():
    rows = []
    n_errors = 0
    for json_path in (DATA / "raw").glob("*.json"):
        with open(json_path, encoding="utf-8") as f:
            data = json.load(f)
        if "error" in data:
            n_errors += 1
            continue
        for song in data.get("songs", []):
            rows.append({
                "artist_name": data["name"],
                "artist_slug": json_path.stem,
                "genius_artist_id": data.get("genius_id"),
                "song_id": song["song_id"],
                "title": song["title"],
                "lyrics": song["lyrics"],
                "url": song["url"],
                "featured_artists": "; ".join(song["featured_artists"]),
            })
    print(f"Loaded {len(rows)} songs from successful scrapes ({n_errors} artists had errors)")
    return pd.DataFrame(rows)


songs_raw = load_scraped_songs()
print(f"Unique artists with songs: {songs_raw['artist_slug'].nunique()}")
songs_raw.head()

Loaded 1499 songs from successful scrapes (1 artists had errors)
Unique artists with songs: 35


,artist_name,artist_slug,genius_artist_id,song_id,title,lyrics,url,featured_artists
0,La Materialista,la_materialista,456495,3570081,Niveles,Si la gente me tiene envidia\nNo les pares que...,https://genius.com/La-materialista-niveles-lyrics,
1,La Materialista,la_materialista,456495,1966299,La Chapa Que Vibran,"¿Aló?\nPapi, ¿te gustan la' chapa' que vibran?...",https://genius.com/La-materialista-la-chapa-qu...,
2,La Materialista,la_materialista,456495,3522039,Te Lo Compré,"Entonce', yo no soy el hombre de la casa\nSí, ...",https://genius.com/La-materialista-te-lo-compr...,Chimbala
3,La Materialista,la_materialista,456495,3103575,Taka Taka,Mami Cuidate que en la calle\nLos perros están...,https://genius.com/La-materialista-taka-taka-l...,Nfasis
4,La Materialista,la_materialista,456495,3829156,Nunca Maniqui,(Con Light G)\nYa me dieron lu' de que tú no t...,https://genius.com/La-materialista-nunca-maniq...,


---
# Phase 5 — Language filtering with fastText

In [22]:
import urllib.request

FASTTEXT_MODEL_PATH = DATA / "lid.176.ftz"
FASTTEXT_URL = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz"

if not FASTTEXT_MODEL_PATH.exists():
    print("Downloading fastText language model (~1MB)...")
    urllib.request.urlretrieve(FASTTEXT_URL, FASTTEXT_MODEL_PATH)
    print(f"Saved to {FASTTEXT_MODEL_PATH}")
else:
    print(f"Model already present at {FASTTEXT_MODEL_PATH}")

import fasttext
fasttext.FastText.eprint = lambda *a, **k: None  # Suppress deprecation warning
ft_model = fasttext.load_model(str(FASTTEXT_MODEL_PATH))
print("fastText model loaded")

Model already present at data/spanish/lid.176.ftz
fastText model loaded


In [23]:
def clean_lyrics(text):
    """Light cleaning before language ID."""
    if not text:
        return ""
    text = re.sub(r"\[[^\]]+\]", "", text)  # Strip [Verse], [Chorus] etc.
    text = re.sub(r"^\d+\s+Contributors.*?Lyrics", "", text, flags=re.DOTALL)  # Genius header
    text = re.sub(r"\d*Embed\s*$", "", text)  # Genius footer
    text = re.sub(r"\s+", " ", text).strip()
    return text


def analyze_song_language(text, ft):
    """Per-song language stats based on line-level classification."""
    cleaned = clean_lyrics(text)
    lines = [ln.strip() for ln in re.split(r"[\n.!?]", cleaned) if ln.strip() and len(ln.strip()) > 3]
    if not lines:
        return {"n_lines": 0, "english_ratio": 0, "l1_ratio": 0, "top_lang": None}
    labels, _ = ft.predict(lines, k=1)
    lang_codes = [lbl[0].replace("__label__", "") for lbl in labels]
    lang_counts = Counter(lang_codes)
    total = len(lang_codes)
    return {
        "n_lines": total,
        "english_ratio": lang_counts.get("en", 0) / total,
        "l1_ratio": lang_counts.get(CONFIG["l1_fasttext_code"], 0) / total,
        "top_lang": lang_counts.most_common(1)[0][0],
    }


lang_stats = []
for _, row in tqdm(songs_raw.iterrows(), total=len(songs_raw), desc="Language ID"):
    lang_stats.append(analyze_song_language(row["lyrics"], ft_model))

lang_df = pd.DataFrame(lang_stats)
songs_with_lang = pd.concat([songs_raw.reset_index(drop=True), lang_df], axis=1)
songs_with_lang["clean_lyrics"] = songs_with_lang["lyrics"].apply(clean_lyrics)
songs_with_lang.to_csv(DATA / "interim" / "songs_with_language.csv", index=False)

print(f"\nSongs analyzed: {len(songs_with_lang)}")
print("\nTop language distribution:")
print(songs_with_lang["top_lang"].value_counts().head(10))

Language ID: 100%|████████████████████████████████████████████████████████████████| 1499/1499 [00:00<00:00, 4887.64it/s]



Songs analyzed: 1499

Top language distribution:
top_lang
es    1285
fr      85
en      80
pt       3
pl       3
de       2
ca       1
sk       1
cs       1
it       1
Name: count, dtype: int64


In [24]:
# Apply thresholds
qualifying = songs_with_lang[
    (songs_with_lang["english_ratio"] >= CONFIG["min_english_ratio"]) &
    (songs_with_lang["n_lines"] >= CONFIG["min_lines_per_song"])
].copy()

borderline = songs_with_lang[
    (songs_with_lang["english_ratio"] >= CONFIG["borderline_ratio"]) &
    (songs_with_lang["english_ratio"] < CONFIG["min_english_ratio"]) &
    (songs_with_lang["n_lines"] >= CONFIG["min_lines_per_song"])
].copy()

qualifying.to_csv(DATA / "interim" / "songs_english.csv", index=False)
borderline.to_csv(DATA / "interim" / "songs_borderline_review.csv", index=False)

print(f"Qualifying (english_ratio >= {CONFIG['min_english_ratio']}): {len(qualifying)} songs by {qualifying['artist_slug'].nunique()} artists")
print(f"Borderline (manual review): {len(borderline)} songs")

Qualifying (english_ratio >= 0.5): 14 songs by 7 artists
Borderline (manual review): 19 songs


---
# Phase 6 — Metadata filtering

In [25]:
COVER_PATTERNS = [
    r"\(cover\)", r"\[cover\]", r"- cover$",
    r"\(remix\)", r"\[remix\]",
    r"\(live\)", r"\[live\]",
    r"\(acoustic\)", r"\(demo\)", r"\(instrumental\)",
]


def is_likely_cover_or_variant(title):
    if not isinstance(title, str):
        return False
    t = title.lower()
    return any(re.search(p, t) for p in COVER_PATTERNS)


def lyrics_fingerprint(text):
    if not isinstance(text, str):
        return ""
    t = re.sub(r"[^\w\s]", "", text.lower())
    t = re.sub(r"\s+", " ", t).strip()
    return t[:500]


filtered = qualifying.copy()
filtered["has_features"] = filtered["featured_artists"].str.len() > 0
filtered["is_cover_variant"] = filtered["title"].apply(is_likely_cover_or_variant)
filtered["fingerprint"] = filtered["clean_lyrics"].apply(lyrics_fingerprint)

before = len(filtered)
filtered = filtered[~filtered["is_cover_variant"]]
print(f"Dropped covers/variants: {before - len(filtered)} → {len(filtered)} remaining")

before = len(filtered)
filtered = filtered.drop_duplicates(subset=["artist_slug", "fingerprint"])
print(f"Dropped lyrics duplicates: {before - len(filtered)} → {len(filtered)} remaining")

print(f"\nSongs with features (flagged, not dropped): {filtered['has_features'].sum()}")

filtered.to_csv(DATA / "interim" / "songs_filtered.csv", index=False)

Dropped covers/variants: 0 → 14 remaining
Dropped lyrics duplicates: 0 → 14 remaining

Songs with features (flagged, not dropped): 4


---
# Handoff to Phase 7 (manual L1 verification)

Two outputs:
1. **`artists_for_verification.csv`** — one row per artist with metadata, plus blank columns for your manual judgments
2. **`songs_filtered.csv`** — the song-level corpus, automatically filtered

Open the artists CSV, fill in `L1_confirmed`, `decision_include_exclude`, etc., then back-filter songs by your verified artist list.

In [27]:
def slugify(name):
    s = re.sub(r"[^\w\s-]", "", str(name).lower()).strip()
    return re.sub(r"[-\s]+", "_", s)

artist_summary = (
    filtered.groupby("artist_slug")
    .agg(
        artist_name=("artist_name", "first"),
        n_qualifying_songs=("song_id", "count"),
        n_with_features=("has_features", "sum"),
        mean_english_ratio=("english_ratio", "mean"),
        total_lines=("n_lines", "sum"),
    )
    .reset_index()
    .sort_values("n_qualifying_songs", ascending=False)
)

# Merge in candidate metadata for verification context
candidate_lookup = candidates.copy()
candidate_lookup["artist_slug"] = candidate_lookup["name"].apply(slugify)
artist_summary = artist_summary.merge(
    candidate_lookup[["artist_slug", "birthplace", "birth_year", "genre", "country", "enwiki_url"]],
    on="artist_slug", how="left"
)

artist_summary["meets_song_minimum"] = artist_summary["n_qualifying_songs"] >= CONFIG["min_songs_per_artist"]

# Blank columns for your manual verification
for col in ["L1_confirmed", "raised_in", "age_moved_if_applicable",
            "L1_confidence_high_med_low", "verification_sources",
            "decision_include_exclude", "notes"]:
    artist_summary[col] = ""

artist_summary.to_csv(
    DATA / "processed" / f"artists_for_verification_{CONFIG['L1']}.csv",
    index=False
)

print(f"Artists ready for manual verification: {len(artist_summary)}")
print(f"  Meeting song minimum ({CONFIG['min_songs_per_artist']}+ songs): {artist_summary['meets_song_minimum'].sum()}")
artist_summary.head(15)

Artists ready for manual verification: 7
  Meeting song minimum (1+ songs): 7


,artist_slug,artist_name,n_qualifying_songs,n_with_features,mean_english_ratio,total_lines,birthplace,birth_year,genre,country,enwiki_url,meets_song_minimum,L1_confirmed,raised_in,age_moved_if_applicable,L1_confidence_high_med_low,verification_sources,decision_include_exclude,notes
0,arca,Arca,3,0,1.000000,62,Caracas,1989.0,experimental music; neoperreo,ES,https://en.wikipedia.org/wiki/Arca_(musician),True,,,,,,,
1,juan_magán,Juan Magán,3,0,0.710928,46,Badalona,1978.0,electro latino; house music,ES,https://en.wikipedia.org/wiki/Juan_Mag%C3%A1n,True,,,,,,,
2,candyman,CandyMan,2,0,0.972222,29,Santiago de Cuba,NaN,NaN,CU,https://en.wikipedia.org/wiki/Candyman_(singer),True,,,,,,,
3,noztra,Noztra,2,2,0.550595,68,NaN,1982.0,reggaeton,NaN,https://en.wikipedia.org/wiki/Noztra,True,,,,,,,
4,tote_king,Tote King,2,1,0.601810,30,Seville,1978.0,hip-hop,NaN,https://en.wikipedia.org/wiki/Tote_King,True,,,,,,,
5,arianna_puello,Arianna Puello,1,0,0.680000,25,San Pedro de Macorís,1977.0,rapping,ES,https://en.wikipedia.org/wiki/Arianna_Puello,True,,,,,,,
6,mala_rodríguez,Mala Rodríguez,1,1,0.696970,66,Jerez de la Frontera,1979.0,Latin hip-hop; alternative hip-hop; flamenco; ...,ES,https://en.wikipedia.org/wiki/Mala_Rodr%C3%ADguez,True,,,,,,,


In [28]:
# Pipeline summary
print("=" * 60)
print(f"PIPELINE SUMMARY — L1: {CONFIG['L1']}")
print("=" * 60)
print(f"Phase 2 candidates (merged):     {len(candidates):>6}")
print(f"Phase 3 pre-filter survivors:    {len(pre_filtered):>6}")
print(f"Phase 4 scraped artists:         {songs_raw['artist_slug'].nunique():>6}")
print(f"Phase 4 scraped songs (raw):     {len(songs_raw):>6}")
print(f"Phase 5 english songs:           {len(qualifying):>6}")
print(f"Phase 6 after metadata filter:   {len(filtered):>6}")
print(f"Artists w/ {CONFIG['min_songs_per_artist']}+ qualifying songs:   {artist_summary['meets_song_minimum'].sum():>6}")
print("=" * 60)
print(f"\nNext step: open {DATA / 'processed' / 'artists_for_verification.csv'}")
print("and manually verify L1 for each artist (Phase 7).")

PIPELINE SUMMARY — L1: spanish
Phase 2 candidates (merged):       4220
Phase 3 pre-filter survivors:        44
Phase 4 scraped artists:             35
Phase 4 scraped songs (raw):       1499
Phase 5 english songs:               14
Phase 6 after metadata filter:       14
Artists w/ 1+ qualifying songs:        7

Next step: open data/spanish/processed/artists_for_verification.csv
and manually verify L1 for each artist (Phase 7).


## English-line extraction

The original criterion (whole songs with ≥95% English content) yielded
only 2 qualifying songs, too few for meaningful classification.
Inspection of the candidate pool confirmed that complete
English tracks by Spanish-L1 artists are rare; most
artists who produce English content do so in **code-switched** form
(English verses interleaved with Spanish hooks, or proper nouns).

Rather than abandon the research question, we shifted the unit of
analysis: instead of treating each song as binary English/Spanish,
we **extract English-only lines from each scraped track** using
line-level fastText classification (`lid.176`, confidence threshold 0.7).
Spanish lines are dropped entirely. Each surviving "document" is the
concatenation of confidently-English lines from one song.

Songs yielding fewer than 100 English-classified words are dropped as
too thin for stable feature extraction.

We considered a second-stage token-level filter to remove residual
Spanish words within English lines,
but rejected this approach because it destroyed sentence structure 
fastText drops not only Spanish words but also slang, proper nouns, and
short tokens it can't classify confidently. The resulting fragments
("I let the out", "is with the") lost the syntactic patterns that NLI
relies on. We also considered masking with `[MASK]` tokens but rejected
this since `[MASK]` would introduce a learned signal in any
transformer-based classifier, leaking class membership rather than
preserving L1 transfer information.

We retain complete extracted lines instead. Residual code-switched
lexical items constitute <5% of corpus tokens and do not provide a
useful classification signal independent of grammatical structure.

We manually inspected several samples and confirmed that:
- Extracted lines read as fluent English with grammar intact
- Spanish lines were correctly dropped at the line level
- Code-switching residue is minimal and lexical, not syntactic

In [30]:
import re
from tqdm import tqdm

def extract_english_lines(raw_text, ft, min_confidence=0.7, min_line_length=5):
    """Return only the lines classified as English from a song.
    
    Args:
        raw_text: Original lyrics with newlines preserved
        ft: loaded fastText model
        min_confidence: Minimum probability for English classification (0-1)
        min_line_length: Skip lines shorter than this many characters
    
    Returns:
        dict with: english_only_text, n_english_lines, n_total_lines, english_ratio
    """
    if not raw_text or not isinstance(raw_text, str):
        return {
            "english_only_text": "",
            "n_english_lines": 0,
            "n_total_lines": 0,
            "english_ratio": 0,
        }
    
    # Strip section headers and Genius boilerplate, keep newlines
    text = re.sub(r"\[[^\]]+\]", "", raw_text)
    text = re.sub(r"^\d+\s+Contributors.*?Lyrics", "", text, flags=re.DOTALL)
    text = re.sub(r"\d*Embed\s*$", "", text)
    
    # Split on actual newlines
    lines = [ln.strip() for ln in text.split("\n") if ln.strip() and len(ln.strip()) >= min_line_length]
    
    if not lines:
        return {
            "english_only_text": "",
            "n_english_lines": 0,
            "n_total_lines": 0,
            "english_ratio": 0,
        }
    
    # Predict per line
    labels, probs = ft.predict(lines, k=1)
    
    english_lines = []
    for line, lbl, prob in zip(lines, labels, probs):
        is_english = lbl[0] == "__label__en"
        confident = prob[0] >= min_confidence
        if is_english and confident:
            english_lines.append(line)
    
    return {
        "english_only_text": "\n".join(english_lines),
        "n_english_lines": len(english_lines),
        "n_total_lines": len(lines),
        "english_ratio": len(english_lines) / len(lines) if lines else 0,
    }


# Apply to all scraped songs (use songs_raw with original lyrics, not cleaned)
print("Extracting English-only lines from each song...")
extracted = []
for _, row in tqdm(songs_raw.iterrows(), total=len(songs_raw)):
    extracted.append(extract_english_lines(row["lyrics"], ft_model))

extract_df = pd.DataFrame(extracted)

# Combine with song metadata
songs_english_only = pd.concat([songs_raw.reset_index(drop=True), extract_df], axis=1)

# Add word count of extracted English text
songs_english_only["english_word_count"] = songs_english_only["english_only_text"].str.split().str.len()

# Save full results before filtering (for debugging)
songs_english_only.to_csv(DATA / "interim" / "songs_english_extracted.csv", index=False)

# Filter to songs with meaningful English content
MIN_ENGLISH_WORDS = 100  # 100 words is roughly a verse

usable = songs_english_only[
    songs_english_only["english_word_count"] >= MIN_ENGLISH_WORDS
].copy()

print(f"\n{'='*60}")
print(f"RESULTS")
print(f"{'='*60}")
print(f"Total scraped songs: {len(songs_english_only)}")
print(f"Songs with >={MIN_ENGLISH_WORDS} English words: {len(usable)}")
print(f"Unique artists: {usable['artist_slug'].nunique()}")
print(f"\nDistribution of English word counts:")
print(songs_english_only["english_word_count"].describe())

print(f"\nTop 20 songs by English word count:")
print(usable.sort_values("english_word_count", ascending=False)[
    ["artist_name", "title", "n_english_lines", "n_total_lines", "english_word_count"]
].head(20).to_string())

# Save the usable corpus
usable.to_csv(DATA / "processed" / "english_extracted_corpus_spanish.csv", index=False)
print(f"\nSaved to: {DATA / 'processed' / 'english_extracted_corpus_spanish.csv'}")

Extracting English-only lines from each song...


100%|█████████████████████████████████████████████████████████████████████████████| 1499/1499 [00:00<00:00, 4590.22it/s]


RESULTS
Total scraped songs: 1499
Songs with >=100 English words: 63
Unique artists: 13

Distribution of English word counts:
count    1499.000000
mean       12.884590
std        53.038736
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       671.000000
Name: english_word_count, dtype: float64

Top 20 songs by English word count:
     artist_name                                   title  n_english_lines  n_total_lines  english_word_count
812     CandyMan                      Melt In Your Mouth               79             90                 671
815     CandyMan                           Playin’ On Me               66             69                 609
811     CandyMan                          Knockin’ Boots               56             66                 486
814     CandyMan              Knockin’ Boots (Radio Mix)               68             86                 467
817     CandyMan                           Today’s Topic               51         

## Final corpus cleanup

The line-extraction step in the previous cell saved a CSV containing
all of its working columns, including the original Spanish `lyrics`
field, which is not part of the final corpus and was causing confusion
(Spanish text appeared to be in the "English corpus").

Here we drop unused columns and keep only what's needed for downstream
modeling:

- `artist_slug`, `artist_name` — identifiers
- `song_id`, `title`, `url` — provenance for each document
- `text` (renamed from `english_only_text`) — the actual corpus content
- `word_count`, `english_ratio`, `n_english_lines`, `n_total_lines` — useful metadata for analysis
- `featured_artists` — flagged for L1 verification (Phase 7)

In [35]:
import pandas as pd

# Load from the INTERIM file (this still has the full text and word count columns)
songs = pd.read_csv(DATA / "interim" / "songs_english_extracted.csv")
print(f"Interim file columns: {list(songs.columns)}")
print(f"Interim rows: {len(songs)}")

# Filter to qualifying songs
MIN_WORDS = 100
usable = songs[songs["english_word_count"] >= MIN_WORDS].copy()
print(f"\nSongs with >={MIN_WORDS} English words: {len(usable)}")

# Keep only what's needed
keep_columns = [
    "artist_slug",
    "artist_name",
    "song_id",
    "title",
    "url",
    "english_only_text",
    "english_word_count",
    "english_ratio",
    "n_english_lines",
    "n_total_lines",
    "featured_artists",
]
keep_columns = [c for c in keep_columns if c in usable.columns]
df_clean = usable[keep_columns].copy()

# Rename for clarity
df_clean = df_clean.rename(columns={
    "english_only_text": "text",
    "english_word_count": "word_count",
})

# Save to the processed folder
df_clean.to_csv(DATA / "processed" / "english_extracted_corpus_spanish.csv", index=False)

print(f"\nCleaned columns: {list(df_clean.columns)}")
print(f"Final corpus: {len(df_clean)} songs by {df_clean['artist_slug'].nunique()} artists")

# Sanity check: peek at first row's text
print(f"\nFirst song text (first 300 chars):")
print(df_clean.iloc[0]["text"][:300])

Interim file columns: ['artist_name', 'artist_slug', 'genius_artist_id', 'song_id', 'title', 'lyrics', 'url', 'featured_artists', 'english_only_text', 'n_english_lines', 'n_total_lines', 'english_ratio', 'english_word_count']
Interim rows: 1499

Songs with >=100 English words: 63

Cleaned columns: ['artist_slug', 'artist_name', 'song_id', 'title', 'url', 'text', 'word_count', 'english_ratio', 'n_english_lines', 'n_total_lines', 'featured_artists']
Final corpus: 63 songs by 13 artists

First song text (first 300 chars):
She give it up when me call her up
She run away even when I stuck around
And she don't want no other man but me
Number one, she one of a kind
She know how she blew my mind
Girl, you and me agree (agree)
When you're not around me, bring me contigo
When you're not around me, bring me contigo
When you'
